## ETL Silver – History (diario y horario)

### Propósito
Transformar los JSON raw históricos (capa Bronze) en tablas Delta Silver:
- una tabla **diaria** (1 fila por `city + date`)
- una tabla **horaria** (1 fila por `city + datetime`)
Ambas quedan deduplicadas por `ingestion_time` (última ingesta disponible).

### Entrada
- Ruta Bronze:
  - `/Volumes/workspace/default/bronce_clima/history/`
- Formato: JSON con payload en `data.forecast.forecastday` + `metadata.ingestion_time`

---

## 1) Tabla Silver diaria: `weather_daily_silver`

### Transformaciones principales
- Explosión del array:
  - `explode(data.forecast.forecastday)` → 1 fila por día
- Selección de métricas desde `forecastday.day`:
  - `maxtemp_c`, `mintemp_c`, `avgtemp_c`, `avghumidity`, `totalprecip_mm`,
    `maxwind_kph`, `daily_chance_of_rain`, `condition.text`
- Tipado:
  - `date` se castea a `date`
- Deduplicación:
  - `row_number() over (partition by city, date order by ingestion_time desc)` y se conserva `row_num = 1`

### Salida
- Tabla Delta (managed): `weather_daily_silver`
- Granularidad: `city + date`

---

## 2) Tabla Silver horaria: `weather_hourly_silver`

### Transformaciones principales
- Explosión de horas:
  - `explode(forecastday.hour)` → 1 fila por hora
- Selección de métricas horarias:
  - `datetime` (desde `hour.time`), `temp_c`, `humidity`, `precip_mm`, `wind_kph`,
    `cloud`, `condition.text`
- Tipado:
  - `datetime` se castea a `timestamp`
  - `date` se castea a `date`
- Deduplicación:
  - `row_number() over (partition by city, datetime order by ingestion_time desc)` y se conserva `row_num = 1`

### Salida
- Tabla Delta (managed): `weather_hourly_silver`
- Granularidad: `city + datetime`

---

### Notas
- La deduplicación asegura 1 fila por clave lógica, conservando el registro con `ingestion_time` más reciente.
- La tabla horaria se usa luego para agregados diarios (picos de lluvia/viento y horas fúngicas) en Gold.


In [0]:
from pyspark.sql.functions import col,explode,lower, regexp_replace,desc,row_number
from delta.tables import DeltaTable
from pyspark.sql.window import Window

In [0]:
df_history_bronze = spark.read.json("/Volumes/workspace/default/bronce_clima/history/")

In [0]:
df_history_bronze.select("city").distinct().show()

In [0]:
df_days = df_history_bronze.select(
    col("city"),
    col("metadata.ingestion_time"),
    explode("data.forecast.forecastday").alias("day")
)


In [0]:
df_days.printSchema()

In [0]:
df_daily = df_days.select(
    col("city"),
    col("ingestion_time"),
    col("day.date").alias("date"),
    col("day.day.maxtemp_c").alias("maxtemp_c"),
    col("day.day.mintemp_c").alias("mintemp_c"),
    col("day.day.avgtemp_c").alias("avgtemp_c"),
    col("day.day.avghumidity").alias("avghumidity"),
    col("day.day.totalprecip_mm").alias("totalprecip_mm"),
    col("day.day.maxwind_kph").alias("maxwind_kph"),
    col("day.day.daily_chance_of_rain").alias("daily_chance_of_rain"),
    col("day.day.condition.text").alias("condition")
).withColumn(
    "date", col("date").cast("date")
)

In [0]:
df_daily.show(5)
df_daily.printSchema()


In [0]:
window_spec = Window.partitionBy("city", "date").orderBy(desc("ingestion_time"))

df_daily = df_daily.withColumn(
    "row_num",
    row_number().over(window_spec)
).filter(
    col("row_num") == 1
).drop("row_num")

In [0]:
df_daily.show(10)

In [0]:
df_daily.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("weather_daily_silver")

In [0]:
spark.table("weather_daily_silver").show(10)

### Tabla HOURLY
#### 1 fila = 1 hora ✅


In [0]:
df_hours = df_days.select(
    "city",
    "ingestion_time",
    "day.date",
    explode("day.hour").alias("hour")
)

In [0]:
df_hours.show(5)
df_hours.printSchema()

In [0]:
df_hourly = df_hours.select(
    "city",
    "ingestion_time",
    col("date").cast("date").alias("date"),
    col("hour.time").alias("datetime"),
    col("hour.temp_c"),
    col("hour.humidity"),
    col("hour.precip_mm"),
    col("hour.wind_kph"),
    col("hour.cloud"),
    col("hour.condition.text").alias("condition")
)


In [0]:
df_hourly.show(50)


In [0]:
df_hourly = df_hourly.withColumn(
    "datetime",
    col("datetime").cast("timestamp")
)

In [0]:
window_spec = Window.partitionBy("city", "datetime").orderBy(desc("ingestion_time"))

df_hourly = df_hourly.withColumn(
    "row_num",
    row_number().over(window_spec)
).filter(
    col("row_num") == 1
).drop("row_num")

In [0]:
df_hourly.show(50)

In [0]:
df_hourly.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("weather_hourly_silver")

In [0]:
spark.table("weather_hourly_silver").show(5)